In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# import random
# random.random()

In [ ]:
import pandas as pd
import numpy as np
import nltk

from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('vader_lexicon')
nltk.download('vader_lexicon')

In [ ]:
# !pip install pycaret

In [ ]:
#Read the dataset
df = pd.read_csv('sample_data/Tweets.csv')
df.head(8)

In [ ]:
df.airline_sentiment.value_counts(normalize=True)

In [ ]:
df.shape

In [ ]:
#Using the tweets to predict the sentiment
#Only using a subset of the data (5000 samples randomly picked)

indexes = list(range(df.shape[0]))
#Taking 5000 samples at random
# l = np.random.choice(indexes,5000)
l = np.random.choice(indexes,df.shape[0])

X = df.loc[l,'text']  #Tweets

Y = df.loc[l,'airline_sentiment'] #The Actual (Target) Sentiments

In [ ]:
len(l)

In [ ]:
"Howdy, my name is Andrew. How are you doing?".split(' ')

In [ ]:
#Let's generate the BoW representation for the tweets
from nltk import word_tokenize, TweetTokenizer

tokenized_tweets = []
for each in X.str.lower():
    tokenized_tweets.append(nltk.word_tokenize(each))

# using the tweet tokenizer
# tokenized_tweets = []
# for each in X.str.lower():
#     tokenized_tweets.append(TweetTokenizer().tokenize(each))

In [ ]:
text = 'Howdy! How are yall :-)'
# TweetTokenizer().tokenize(text)
word_tokenize(text)

In [ ]:
tokenized_tweets[:2] #First two tokenized tweets

In [ ]:
#Next, we will remove the stopwords
#Let's fetch the English language stopwords
from nltk.corpus import stopwords
sw_list = stopwords.words('english')
print(sw_list)

sw_list.extend(['@',"'",'.','"','/','!',',',"'ve","...","n't",'$',"'s",'''"''',"''",'..','&','*',';','”','``',':','#','!','-','?','%',"'d","'m",'+','++'])

In [ ]:
tweets_after_removing_SW = []
for each in tokenized_tweets:
    line = []
    for word in each:
        if word not in sw_list:
            line.append(word)
    tweets_after_removing_SW.append(line)

tweets_after_removing_SW[:2] #First 2 tweets after removal of Stopwords
#print(tweets_after_removing_SW)

In [ ]:
#Andrew -> and ##rew

In [ ]:
#Let's do stemming to reduce the number of features!
from nltk.stem import SnowballStemmer
s_stemmer = SnowballStemmer('english')

s_stemmed_list = [] #After Stemming

for each_list in tweets_after_removing_SW:
    line = []
    for word in each_list:
        line.append(s_stemmer.stem(word))
    s_stemmed_list.append(line)
s_stemmed_list[:2]

In [ ]:
# s_stemmed_list[:5]

# Using BoW representation

In [ ]:
# s_stemmed_list

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
inp = pd.DataFrame(te.fit(s_stemmed_list).transform(s_stemmed_list).astype(int), columns = te.columns_)
inp.head(8)

In [ ]:
inp.shape

In [ ]:
len(s_stemmed_list)

In [ ]:
# for sentence in s_stemmed_list[:20]:
#   print(len(sentence))
# prob(pos|word1, word2, word3)

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(inp, Y, test_size = 0.3, random_state = 2)

In [ ]:
from sklearn.naive_bayes import GaussianNB
model = GaussianNB()
model.fit(x_train, y_train)

In [ ]:
y_hat = model.predict(x_test) #Predictions
y_hat

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, y_hat)

Accuracy = $\frac{TP + TN}{P + N} = \frac{TP + TN}{TP + TN + FP + FN}$

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from matplotlib import pyplot as plt
disp = ConfusionMatrixDisplay(confusion_matrix(y_test, y_hat), display_labels=list(np.unique(Y)))
disp.plot()
plt.show()

## Using Term Frequency Matrix

In [ ]:
# Convert List of lists to list of Strings
# using map() + join()
str_data = list(map(' '.join, s_stemmed_list))
str_data[:5] # Displaying the top 5 stemmed strings

In [ ]:
# s_stemmed_list[0]

'This is the first document.'

'This document is the second document.'

'And this is the third one.'

'Is this the first document?'

['and', 'document', 'first', 'is', 'one', 'second', 'the', 'third', 'this']

[
  
  [0 1 1 1 0 0 1 0 1]

 [0 2 0 1 0 1 1 0 1]

 [1 0 0 1 1 0 1 1 1]

 [0 1 1 1 0 0 1 0 1]

 ]

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()
cv.fit(str_data)
str_df = pd.DataFrame(cv.transform(str_data).todense(), columns = sorted(cv.vocabulary_))
str_df.head()

In [ ]:
str_df.shape

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(str_df, Y, test_size = 0.3, random_state = 2)

model = GaussianNB()
model.fit(x_train, y_train)

y_hat = model.predict(x_test) #Predictions
accuracy_score(y_test, y_hat)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix(y_test, y_hat), display_labels=list(np.unique(Y)))
disp.plot()
plt.show()

# Using TF-IDF

In [ ]:
# the dog went to the park

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
fidf = TfidfVectorizer()
fidf.fit(str_data)
tfidf_df = pd.DataFrame(fidf.transform(str_data).todense(), columns = sorted(fidf.vocabulary_))
tfidf_df.head()

In [ ]:
# fidf.transform(str_data).todense()

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(tfidf_df, Y, test_size = 0.3, random_state = 2)

model = GaussianNB()
model.fit(x_train, y_train)

y_hat = model.predict(x_test) #Predictions
accuracy_score(y_test, y_hat)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix(y_test, y_hat), display_labels=list(np.unique(Y)))
disp.plot()
plt.show()

In [ ]:
# tweet = united is awesome
# wv.(united) -> [0.988, ....., 0.653] (length (100))
# wv.(is) -> [0.976, ....., 0.352]
# wv.(awesome) -> [0.238, ....., 0.233]

# tweet = (wv.(united) + wv.(is) + wv.(awesome))/num_tokens_tweet = [......] length = 100

> To improve these results, you can think of reducing the dimensions of the input data by using appropriate dimensionality reduction techniques (feature selection & feature extraction).

> You can explore Ensemble-based classification models for better baseline results.

> Explore pre-trained models that have been built specifically for sentiment classification.

> Use sentence level embeddings from pretrained models for this supervised task.  

In [ ]:
# We recommend using PyCaret for testing multiple classifiers simultaneously.
# !pip install pycaret
import pandas as pd
from pycaret.classification import setup, compare_models

In [ ]:
tfidf_df.shape

In [ ]:
len(Y.values)

In [ ]:
# tfidf_df.drop('target', axis= 1, inplace = True)
tfidf_df['Y'] = Y.values# .reshape(-1,1)

# The setup() function must also be called before executing any other function,
# its two mandatory parameters being “data” and “target”, which will be our main column for operation.

#setting the experiment
experiment = setup(tfidf_df.loc[:500,:], target='Y')
# Choosing a small dataset to demonstrate the usage of PyCaret.
# You can run this on Colab to get the results on the complete dataset.

#show the best model and their statistics
best_model = compare_models()

### More details abot the usage of PyCaret can be found here:
https://analyticsindiamag.com/building-an-ml-classification-model-using-pycaret/

> LazyPredict package is another alternative that you can try for comparing the performance of multiple models simultaneously.

# Using Sentence Embeddings
## Training Doc2Vec

In [ ]:
# Do not run ... takes too long.
from gensim.models.doc2vec import Doc2Vec, TaggedDocument

# The input should be passed in sentences (list of strings)
sentences = str_data[:3000] # Using the first 3000 samples to train the model

# Tag the sentences for training
tagged_data = [TaggedDocument(words=sentence.split(), tags=[str(i)]) for i, sentence in enumerate(sentences)]

# Train the model
model = Doc2Vec(tagged_data, vector_size=5, window=2, min_count=3, workers=4, epochs = 100)

In [ ]:
# Get the embeddings for the sentences
sentences = str_data
sentence_vectors = [model.infer_vector(sentence.split()) for sentence in sentences]

# The infer_vectors expects the input as a list of words (nltk.word_tokenize())

print("Sentence Embeddings:")
print(np.array(sentence_vectors).shape) #Embeddings of the sentences

In [ ]:
sentence_vectors[:5]

In [ ]:
x_train = np.array(sentence_vectors)[:3000,:]
y_train = Y[:3000]

x_test =  np.array(sentence_vectors)[3000:,:]
y_test =  Y[3000:]

model = GaussianNB()
model.fit(x_train, y_train)

y_hat = model.predict(x_test) #Predictions
accuracy_score(y_test, y_hat)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix(y_test, y_hat), display_labels=list(np.unique(Y)))
disp.plot()
plt.show()

In [ ]:
# from mlxtend.preprocessing import TransactionEncoder

# dataset = [['Apple', 'Beer', 'Rice', 'Chicken', 'Rice', 'Rice'],
#            ['Apple', 'Beer', 'Rice'],
#            ['Apple', 'Beer'],
#            ['Apple', 'Bananas'],
#            ['Milk', 'Beer', 'Rice', 'Chicken'],
#            ['Milk', 'Beer', 'Rice'],
#            ['Milk', 'Beer'],
#            ['Apple', 'Bananas']]

# te = TransactionEncoder()
# te_ary = te.fit(dataset).transform(dataset)
# te_ary

# import pandas as pd

# pd.DataFrame(te_ary, columns=te.columns_)